# unlearn-quant on Kaggle (T4)
Settings: **GPU T4**, **Internet ON**. The harness reaches this notebook either by (a) git clone of your repo, or (b) attaching the `unlearn-quant` folder as a Kaggle Dataset.

Phase A = replication (bnb only). Phase B = edge int4 + distillation.

In [ ]:
# 1. Get the harness into /kaggle/working
import os, shutil, subprocess
REPO_URL = os.environ.get('UNLEARN_QUANT_REPO', 'https://github.com/Arya-126/unlearn-quant.git')  # override via env, or blank to use an attached dataset
WORK = '/kaggle/working/unlearn-quant'
if REPO_URL:
    subprocess.run(['git','clone',REPO_URL,WORK], check=True)
elif os.path.isdir('/kaggle/input/unlearn-quant'):
    shutil.copytree('/kaggle/input/unlearn-quant', WORK, dirs_exist_ok=True)
else:
    raise SystemExit('Provide UNLEARN_QUANT_REPO or attach the unlearn-quant dataset.')
os.chdir(WORK)
print('cwd', os.getcwd(), os.listdir())

In [ ]:
# 2. Install ONLY missing extras. DO NOT touch torch/transformers/numpy on Kaggle —
#    reinstalling torch from PyPI pulls a CPU-only wheel and kills the GPU.
import importlib.util as u, subprocess, sys

def ensure(pkg, import_name=None):
    if u.find_spec(import_name or pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", "--no-deps", pkg], check=True)

ensure("rouge_score", "rouge_score")
ensure("absl-py", "absl"); ensure("nltk", "nltk")   # rouge_score runtime deps (no torch/numpy)
ensure("bitsandbytes", "bitsandbytes")
ensure("peft", "peft")

# Hard gate: confirm the CUDA torch is intact before spending a session.
import torch
assert torch.cuda.is_available(), (
    "CUDA torch missing! The kernel likely had torch reinstalled. "
    "Factory-reset the session (no -r requirements.txt) and re-run."
)
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))

In [ ]:
# 3. (Phase B ONLY — skip for the Phase A gate.) Edge int4 toolchains.
#    All installs use --no-deps so they cannot upgrade torch/transformers.
#    Failures are non-fatal: per-condition errors are recorded and bnb/fp16 still run.
import os, subprocess

# AWQ (activation-aware). autoawq tracks transformers; --no-deps avoids upgrades.
subprocess.run("pip -q install --no-deps autoawq || echo 'awq install skipped'", shell=True)
# GPTQ. 7.x demands transformers>=5; pin a 4.x-compatible line. May still need tuning.
subprocess.run("pip -q install --no-deps 'gptqmodel<2' tokenicer device-smi logbar || echo 'gptq install skipped'", shell=True)

# GGUF via llama.cpp (heaviest; optional). Build only llama-quantize.
LCPP = '/kaggle/working/llama.cpp'
if not os.path.isdir(LCPP):
    subprocess.run(['git','clone','--depth','1','https://github.com/ggerganov/llama.cpp',LCPP])
    subprocess.run(['cmake','-B',f'{LCPP}/build',LCPP])
    subprocess.run(['cmake','--build',f'{LCPP}/build','--config','Release','-j','2','--target','llama-quantize'])
    subprocess.run(['pip','-q','install','-r',f'{LCPP}/requirements.txt'])
subprocess.run("pip -q install llama-cpp-python || echo 'llama-cpp-python skipped -> GGUF conditions skipped'", shell=True)
os.environ['LLAMACPP_DIR'] = LCPP
print('Phase B deps attempted (check above for any "skipped" lines).')

In [ ]:
# 4a. Phase A — replication gate on BOTH utility-constrained methods (bnb only).
#     Each appends rows to results/summary.csv.
!python -m scripts.run_replication --method graddiff --n-eval 100
!python -m scripts.run_replication --method npo --n-eval 100

In [ ]:
# 4b. Phase B — edge int4 + distillation (after Phase A passes the recovery gate)
!python -m scripts.run_edge_extension --method graddiff --n-eval 100 --distill

In [ ]:
# 5. Surface results (also persisted to /kaggle/working for download)
import pandas as pd
df = pd.read_csv('results/summary.csv')
import shutil; shutil.copytree('results','/kaggle/working/results', dirs_exist_ok=True)
df[['method','condition','forget_score','recovery_ratio','model_utility','forget_quality']]